## Geoginah Ramatshe_26303686

# 4. Model Validation — Unseen Test Period

This notebook loads the saved model pipelines and evaluates them on the later test subset that was not used for training or hyperparameter selection. It reports classification metrics and saves predictions for the results visualisation notebook.

## Evaluation approach

The test period is 31 January to 31 May 2026. Use the same probability threshold (`0.50`) for the class-based metrics so the models can be compared fairly. Accuracy alone can be misleading when classes are imbalanced, so precision, recall, F1, ROC-AUC and PR-AUC are reported as well. PR-AUC focuses on precision and recall for the dormant class.

This notebook refits each available model on the combined training and validation subsets after the XGBoost configuration has been selected. The held-out test data remain separate until final evaluation.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Raw CSV files are expected to be in the same folder as these notebooks.
BASE_DIR = Path.cwd()
RAW_FILES = {
    "profile": "table1_customer_profile.csv",
    "funding": "table2_account_funding.csv",
    "transactions": "table3_transaction_trading.csv",
    "engagement": "table4_platform_engagement.csv",}

def safe_date(series):
    return pd.to_datetime(series, errors="coerce")

missing_files = [name for name in RAW_FILES.values() if not (BASE_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Could not find these CSV files in the notebook's current folder: "
        + ", ".join(missing_files)
        + ". Place the CSV files beside the notebooks or update BASE_DIR.")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report)

import joblib

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "processed"
ARTIFACT_DIR = BASE_DIR / "model_artifacts"
RESULTS_DIR = BASE_DIR / "results"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required = [
    PROCESSED_DIR / "train_data.csv",
    PROCESSED_DIR / "validation_data.csv",
    PROCESSED_DIR / "test_data.csv",]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Prepared data not found. Run 02_dataset_preprocessing.ipynb first. Missing: "
        + ", ".join(missing))

train_data = pd.read_csv(PROCESSED_DIR / "train_data.csv", parse_dates=["snapshot_date"])
validation_data = pd.read_csv(PROCESSED_DIR / "validation_data.csv", parse_dates=["snapshot_date"])
test_data = pd.read_csv(PROCESSED_DIR / "test_data.csv", parse_dates=["snapshot_date"])

target_column = "dormancy_target"
excluded_columns = {
    "customer_id", "snapshot_date", target_column,
    "active_investor", "dormant_investor", "days_since_last_activity",
    "registration_date", "registration_dt", "first_deposit_date", "first_deposit_dt",
    "first_deposit_amount", "last_deposit_date", "total_deposit_amount",
    "number_of_deposits", "funding_status", "kyc_completed",}

feature_columns = [c for c in train_data.columns if c not in excluded_columns]

X_train = train_data[feature_columns].copy()
y_train = train_data[target_column].astype(int).copy()
X_val = validation_data[feature_columns].copy()
y_val = validation_data[target_column].astype(int).copy()
X_test = test_data[feature_columns].copy()
y_test = test_data[target_column].astype(int).copy()

# Standardise missing values in text columns for scikit-learn imputers.
for frame in (X_train, X_val, X_test):
    for col in frame.select_dtypes(include=["object", "string", "category"]).columns:
        frame[col] = frame[col].astype(object).where(frame[col].notna(), np.nan)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()

assert "dormant_investor" not in feature_columns
assert "active_investor" not in feature_columns
assert "days_since_last_activity" not in feature_columns

def make_preprocessor():
    numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                                       ("scaler", StandardScaler()),])
    
    categorical_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
                                           ("onehot", OneHotEncoder(handle_unknown="ignore")),])
    
    return ColumnTransformer(transformers=[("numeric", numeric_pipeline, numeric_features), 
                                           ("categorical", categorical_pipeline, categorical_features),], remainder="drop")

def compute_metrics(y_true, y_probability, threshold=0.50):
    y_probability = np.asarray(y_probability)
    y_prediction = (y_probability >= threshold).astype(int)
    return {
        "instances": int(len(y_true)),
        "dormant_proportion": float(np.mean(y_true)),
        "accuracy": float(accuracy_score(y_true, y_prediction)),
        "precision": float(precision_score(y_true, y_prediction, zero_division=0)),
        "recall": float(recall_score(y_true, y_prediction, zero_division=0)),
        "f1": float(f1_score(y_true, y_prediction, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_probability)) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": float(average_precision_score(y_true, y_probability)),
    }

## Loading the saved Logistic Regression model and refit on train + validation

In [4]:
train_validation_data = pd.concat([train_data, validation_data], ignore_index=True)
X_train_validation = train_validation_data[feature_columns].copy()
y_train_validation = train_validation_data[target_column].astype(int).copy()

for col in X_train_validation.select_dtypes(include=["object", "string", "category"]).columns:
    X_train_validation[col] = X_train_validation[col].astype(object).where(X_train_validation[col].notna(), np.nan)

logistic_model_path = ARTIFACT_DIR / "logistic_model.joblib"
if not logistic_model_path.exists():
    raise FileNotFoundError("Run 03_model_training.ipynb first to create the Logistic Regression model.")
    
logistic_final = joblib.load(logistic_model_path)
logistic_final.fit(X_train_validation, y_train_validation)
logistic_test_probability = logistic_final.predict_proba(X_test)[:, 1]

print(logistic_test_probability)

[0.81811767 0.58780187 0.19106921 ... 0.15499637 0.9778679  0.66381708]


## Refitting XGBoost if its training artefact exists

In [5]:
xgb_available = False
xgb_final = None
xgb_test_probability = None
xgb_model_path = ARTIFACT_DIR / "xgboost_validation_model.joblib"

try:
    from xgboost import XGBClassifier
    xgb_installed = True
except ImportError:
    xgb_installed = False

if xgb_installed and xgb_model_path.exists():
    # Load selected validation pipeline to obtain the selected hyperparameters.
    selected_xgb_pipeline = joblib.load(xgb_model_path)
    selected_params = selected_xgb_pipeline.named_steps["classifier"].get_params()
    positive = max(int(y_train_validation.sum()), 1)
    negative = int((y_train_validation == 0).sum())
    selected_params["scale_pos_weight"] = negative / positive
    selected_params["random_state"] = 42
    selected_params["n_jobs"] = 4

    xgb_final = Pipeline(steps=[("preprocessing", make_preprocessor()),
                                ("classifier", XGBClassifier(**selected_params)),])
    
    xgb_final.fit(X_train_validation, y_train_validation)
    xgb_test_probability = xgb_final.predict_proba(X_test)[:, 1]
    xgb_available = True
else:
    print("XGBoost test evaluation skipped: install xgboost and/or run 03_model_training.ipynb first.")

## Calculating test metrics and save predictions

In [6]:
def report_metrics(name, y_true, probability):
    prediction = (np.asarray(probability) >= 0.50).astype(int)
    print(f"\n{name} — unseen test period")
    print(classification_report(y_true, prediction, digits=3, zero_division=0))
    print("Confusion matrix [rows=actual 0/1, columns=predicted 0/1]:")
    print(confusion_matrix(y_true, prediction, labels=[0, 1]))
    return {"model": name, **compute_metrics(y_true, probability)}

test_rows = [report_metrics("Logistic Regression", y_test, logistic_test_probability)]
if xgb_available:
    test_rows.append(report_metrics("XGBoost", y_test, xgb_test_probability))

test_metrics = pd.DataFrame(test_rows)
display(test_metrics.round(4))
test_metrics.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)

test_predictions = pd.DataFrame({
    "customer_id": test_data["customer_id"].to_numpy(),
    "snapshot_date": test_data["snapshot_date"].to_numpy(),
    "actual": y_test.to_numpy(),
    "logistic_probability": logistic_test_probability,
    "logistic_prediction": (logistic_test_probability >= 0.50).astype(int),
})
if xgb_available:
    test_predictions["xgboost_probability"] = xgb_test_probability
    test_predictions["xgboost_prediction"] = (xgb_test_probability >= 0.50).astype(int)

test_predictions.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
joblib.dump(logistic_final, ARTIFACT_DIR / "logistic_final_model.joblib")
if xgb_available:
    joblib.dump(xgb_final, ARTIFACT_DIR / "xgboost_final_model.joblib")
print("Saved test metrics and predictions to:", RESULTS_DIR.resolve())


Logistic Regression — unseen test period
              precision    recall  f1-score   support

           0      0.875     0.603     0.714      2003
           1      0.366     0.727     0.487       633

    accuracy                          0.632      2636
   macro avg      0.620     0.665     0.600      2636
weighted avg      0.753     0.632     0.659      2636

Confusion matrix [rows=actual 0/1, columns=predicted 0/1]:
[[1207  796]
 [ 173  460]]

XGBoost — unseen test period
              precision    recall  f1-score   support

           0      0.881     0.628     0.733      2003
           1      0.383     0.731     0.503       633

    accuracy                          0.653      2636
   macro avg      0.632     0.679     0.618      2636
weighted avg      0.761     0.653     0.678      2636

Confusion matrix [rows=actual 0/1, columns=predicted 0/1]:
[[1257  746]
 [ 170  463]]


,model,instances,dormant_proportion,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression,2636,0.2401,0.6324,0.3662,0.7267,0.4870,0.7362,0.5464
1,XGBoost,2636,0.2401,0.6525,0.3830,0.7314,0.5027,0.7548,0.6082


Saved test metrics and predictions to: C:\Users\HP\Documents\4.CAPSTONE Project\SS3 final\retail_investor_dormancy_notebooks\results
